## Models
Models can be utilized in two ways:
1. With **_agents_** method - Models can be dynamically specified when creating an agent.
2. Standalone - Models can be called directly (outside of the agent loop) for tasks like text generation, classification, or extraction without the need for an agent framework.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

#### Using Chat Model

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4o")   ## Pass any model name ex: claude-sonnet-4-5-20250929
model.invoke("What's Gen AI")

#### Using Class Model (OpenAI)

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o")
model.invoke("Whats ML?")

#### Using Class Model (Anthropic, Azure, AWS)

In [ ]:
from langchain_openai import AzureChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_aws import ChatBedrock

azure_model = AzureChatOpenAI(model="gpt-4o")
anthropic_model = ChatAnthropic("claude-sonnet-4-5-20250929")
bedrock_model = ChatBedrock(model="claude-sonnet-4-5-20250929")

#### Parameters

In [ ]:
"""
    model (required):
    The name or identifier of the specific model you want to use with a provider. You can also specify both the model and its provider in a single argument using the ’:’ format, for example, ‘openai:o1’.

    api_key:
    The key required for authenticating with the model’s provider.

    temperature:
    Controls the randomness of the model’s output. A higher number makes responses more creative; lower ones make them more deterministic.

    max_tokens:
    Limits the total number of tokens in the response, effectively controlling how long the output can be.

    timeout:
    The maximum time (in seconds) to wait for a response from the model before canceling the request.

    max_retries:
    The maximum number of attempts the system will make to resend a request if it fails due to issues like network timeouts or rate limits.
"""

model = init_chat_model(
    model="gpt-4o",
    temperature=0.7,
    max_tokens=1000,
    timeout=30,
    max_retries=2
)

### Invocation

In [ ]:
"""
    invoke() method
"""
model = init_chat_model(model="gpt-4o")
model.invoke("Whats Gen AI")

In [ ]:
"""
    stream() method
"""
for chunk in model.stream("Whats GenAI?"):
    print(chunk.text, end="", flush=True)

In [ ]:
"""
    batch() - method
"""
responses = model.batch([
    "Whats Gen AI?",
    "Whats ML?",
    "Whats DL?"
])

for response in responses:
    print(response.content)

#### Tools

In [ ]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """ get the weather of the location """
    return f"Weather of {location} is sunny today"


@tool
def greetings(name: str) -> str:
    """ Greet the user by their provided name """
    return f"Greetings, {name}. How are you?"

model = init_chat_model(model="gpt-4o")
model_with_tools = model.bind_tools([get_weather, greetings])

model_with_tools.invoke("What's the weather like in Boston?")

#### Structured Output using Pydantic

In [9]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name: str = Field(..., description="name of the cast member")
    role_name: str = Field(..., description="name of the character in the movie")


class MovieDetails(BaseModel):
    """ A movie with details """
    title: str = Field(..., description="title of the movie")
    year: int = Field(..., description="year of the movie released")
    cast: list[Actor] = Field(None, description="cast members of the movie")

model = init_chat_model(model="gpt-4o")
structured_model = model.with_structured_output(MovieDetails)
response = structured_model.invoke("Give me the details of movie called Fight Club")
print(response)

title='Fight Club' year=1999 cast=[Actor(name='Edward Norton', role_name='The Narrator'), Actor(name='Brad Pitt', role_name='Tyler Durden'), Actor(name='Helena Bonham Carter', role_name='Marla Singer'), Actor(name='Meat Loaf', role_name="Robert 'Bob' Paulson"), Actor(name='Jared Leto', role_name='Angel Face'), Actor(name='Zach Grenier', role_name='Richard Chesler'), Actor(name='Holt McCallany', role_name='The Mechanic')]
